In [1]:
# Parameters
base_path = "C:\\Users\\ander\\OneDrive\\TCC_USP"
run_id = "20251117_163225"


In [2]:
# 1. Setup de caminhos locais
import os
import subprocess
from datetime import datetime
from pathlib import Path

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "05_data_collection_real"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)


BASE_PATH: C:\Users\ander\OneDrive\TCC_USP
PROC_PATH: C:\Users\ander\OneDrive\TCC_USP\data_processed


In [3]:
# Legacy Colab mount removido; paths definidos no setup.


In [4]:
# 3. Buscar notÃ­cias (NewsAPI)
import requests, pandas as pd
from datetime import datetime, timedelta

API_KEY = "3d606b02f24447849f49b3e8c3628f46"
URL = "https://newsapi.org/v2/everything"

params = {
    "q": "Ibovespa OR Bovespa OR Petrobras OR Vale OR dÃ³lar OR economia",
    "language": "pt",
    "sortBy": "publishedAt",
    "pageSize": 100,
    "apiKey": API_KEY
}

resp = requests.get(URL, params=params)
data = resp.json()

noticias = []
for art in data.get("articles", []):
    noticias.append({
        "data": art["publishedAt"][:10],  # formato YYYY-MM-DD
        "fonte": art["source"]["name"],
        "titulo": art["title"],
        "texto": art["description"] or ""
    })

df_new = pd.DataFrame(noticias)
print("Novas notÃ­cias coletadas:", df_new.shape)
display(df_new.head())

# 4. Incrementar dataset
out_file = os.path.join(RAW_PATH, "noticias_real.csv")

if os.path.exists(out_file):
    df_old = pd.read_csv(out_file)
    print("Dataset anterior:", df_old.shape)
else:
    df_old = pd.DataFrame(columns=["data","fonte","titulo","texto"])
    print("Nenhum dataset anterior encontrado, iniciando do zero.")

# Concatenar e remover duplicatas
df_full = pd.concat([df_old, df_new], ignore_index=True)
df_full = df_full.drop_duplicates(subset=["data","titulo"]).reset_index(drop=True)

# Salvar atualizado
df_full.to_csv(out_file, index=False, encoding="utf-8-sig")

print(f"âœ… AtualizaÃ§Ã£o concluÃ­da! Total de notÃ­cias agora: {df_full.shape[0]}")

Novas notÃ­cias coletadas: (87, 4)


,data,fonte,titulo,texto
0,2025-11-16,InfoMoney,"Ibovespa nas alturas, falência da Oi, COP30 e ...",O InfoMoney Resume para você: fique por dentro...
1,2025-11-16,Observador.pt,Reitores pedem aumento de financiamento da inv...,Os reitores apelam a que exista uma reforma da...
2,2025-11-16,Abril.com.br,Pesquisas trazem muitos alertas e um alento pa...,Queda no mau humor com o preço da comida atenu...
3,2025-11-16,Terra.com.br,"Lima, a joia costeira do Peru: história, sabor...","Descubra dicas de turismo em Lima, no Peru, co..."
4,2025-11-16,Eurisko.com.br,Google renova o app Gemini com novo visual e m...,Principais destaques: O Google apresentou uma ...


Dataset anterior: (297, 4)
âœ… AtualizaÃ§Ã£o concluÃ­da! Total de notÃ­cias agora: 298
